# WO-05 — the place layer, framing A, and the WO-04 fixes
Items 1 to 3 of `wo05_place layer-framingA-wo04-fixes.md`. Item 1: the place layer (point elevation, the point's ecoregion, the basin's elevation range) filled at the ten test places and over the 306 sample basins. Item 2: evidence for the water wording of the five classes. Item 3: readable names, the ecoregion-to-Wikipedia audit, Alexandria's karst, the Caspian. The generator builds the filled_profiles from the wording in the live template file `profile_templates.md` (the script's default; the v4 target was swapped in on 2026-11-07). Reads the local server, the local database and the saved WO-03 tables. Read-only except `wo05_filled_profiles.md`, `wo05_wikilinks.csv` and `output/edop/whg/wo05_fill.csv`.

In [1]:
%matplotlib inline
# Cell 1 — Setup: server check, saved ring and flow tables
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
SAMPLE = C.population_points()
print('commit', C.commit(), '| server:', W.get_json('/api/health'), '| wording:', W.templates_stamp(), '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))

commit 4e99a7f | server: {'status': 'ok'} | wording: profile_templates.md v4 (last commit c34cc46; edited since: yes); script wo04_fill.py (last commit c34cc46; edited since: yes) | test places: 10 | sample points: 306


In [2]:
# Cell 2 — Item 1: the place layer at the ten test places
rows, TEST = [], {}
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW)
    U = W.fill_units(F)
    TEST[name] = (F, U)
    pub, g = F['pub'], (lambda u: U[u]['text'] if U.get(u, {}).get('ok') else None)
    mean = conn.execute('SELECT ele_mt_sav::float8 FROM public.basin06 WHERE hybas_id = %s', (F['hid'],)).fetchone()[0] if F['hid'] else None
    rows.append(dict(place=name, elev_m=g('S1.1b'), dem=(f"{pub.get('elev_dataset')} {pub.get('elev_resolution_m')} m" if pub else None),
                     point_eco=F['pt_eco_name'], basin_eco=(W.sigval(pub, 'ecoregion') if pub else None),
                     eco_differs=(bool(g('S1.2c')) if pub else None), basin_min=g('S1.5a'), basin_max=g('S1.5b'),
                     basin_mean=(round(mean) if mean is not None else None)))
print(pd.DataFrame(rows).to_string())

              place elev_m          dem                               point_eco                                basin_eco eco_differs basin_min basin_max  basin_mean
0          Timbuktu    272  mapzen 30 m                 Sahelian Acacia savanna        Inner Niger Delta flooded savanna        True       257       279       266.0
1           Tbilisi    418  mapzen 30 m      Azerbaijan shrub desert and steppe                   Caucasus mixed forests        True       255     3,838      1638.0
2          Santa Fe  2,128  mapzen 30 m             Colorado Plateau shrublands              Colorado Plateau shrublands       False     1,436     3,936      1979.0
3           Kaifeng     73  mapzen 30 m            Huang He Plain mixed forests             Huang He Plain mixed forests       False        15       137        50.0
4              Lviv    283  mapzen 30 m          Central European mixed forests           Central European mixed forests       False       128       476       212.0
5         

In [3]:
# Cell 3 — The ten filled_profiles from the current wording (written to docs/design/dashboard/wo05_filled_profiles.md); prints Tbilisi
OUTMD = ROOT / 'docs' / 'design' / 'dashboard' / 'wo05_filled_profiles.md'
parts = ['# WO-05: the dummy filled from real data, ten test places (filled_profiles)', '',
         'Generated by `notebooks/edop/whg/wo05_place_layer.ipynb` (commit ' + C.commit() + '). Every `yes` unit of `modal_dummy_annotated.md` is filled from data; each `maybe` or `no` unit is a placeholder in the form `\u27e6id maybe: needs ...\u27e7`. Wording is the template file\'s, not polished.', '',
         'Wording from `' + W.templates_stamp() + '`.', '']
for name in C.PLACES:
    F, U = TEST[name]
    parts.append(W.render_filled_profile(name, F, U)); parts.append('---'); parts.append('')
OUTMD.write_text('\n'.join(parts))
print(f'wrote {OUTMD.name}\n\n' + W.render_filled_profile('Tbilisi', *TEST['Tbilisi']))

wrote wo05_filled_profiles.md

## Tbilisi: environmental profile

### Setting
Tbilisi sits at about 418 m above sea level, in a basin of ⟦S1.1c maybe: needs terrain scheme or cutoff (wo04b)⟧ country that runs from about 255 m to 3,838 m. It lies in the Azerbaijan shrub desert and steppe ecoregion, ⟦S1.2b maybe: needs stored one-line ecoregion gist⟧. Most of its basin lies in the Caucasus mixed forests ecoregion. ⟦S1.3 maybe: needs distance from the point to the next ecoregion⟧ ⟦S1.4 maybe: needs share cutoff for further ecoregions⟧

### Climate
Across the surrounding basin of about 23,251 km², rainfall averages about 762 mm a year, ⟦S2.1b maybe: needs rule for "mostly in {months}"⟧. Temperatures there average 5.3 °C, from -5.9 °C in January to 15.8 °C in August. ⟦S2.3 maybe: needs ring rank rules⟧

### Water
Tbilisi is ⟦S3.1 maybe: needs nearest-river layer: distance, name⟧. No other basin drains into this one: the water here is gathered within the basin itself. ⟦S3.3 maybe: needs cuto

In [ ]:
# Cell 4 — The sample: 306 basin interior points, with the place layer and the water class inputs; saved to a CSV
UNITS = ['S1.1b', 'S1.2a', 'S1.2c', 'S1.5a', 'S1.5b', 'S2.1a', 'S2.2a', 'S2.2bc', 'S3.2', 'S3.2a', 'S3.5', 'S4.2', 'S5.1a', 'S5.1b', 'S6.1a', 'S6.2', 'S7.2', 'S8.1a', 'S8.2', 'S9.2']
rows, t_start = [], time.perf_counter()
for k, s in enumerate(SAMPLE, 1):
    F = W.build_facts(float(s['lat']), float(s['lon']), conn, PAIRS, FLOW)
    U = W.fill_units(F)
    cls, pr, ar, tr = W.provenance(F)
    d = dict(core_id=int(s['hybas_id_l6']), zone=s['climate_zone'], cls=cls, pr=pr, ar=ar, tr=tr,
             area_ratio=F['up_area'] / F['sub_area'], up_area=F['up_area'], dist_sink=W.sigval(F['pub'], 'dist_sink'),
             dem=F['pub'].get('elev_dataset'), eco_differs=int(bool(U['S1.2c']['text'])),
             s82_vars=';'.join(r['variable'] for r in F['areas']['rows'] if r['status'] != 'ok'))
    for u in UNITS:
        d[u] = int(U[u]['ok']); d[u + '_na'] = int(U[u].get('na', False)); d[u + '_why'] = U[u]['why'] or ''
    rows.append(d)
    if k % 50 == 0:
        print(f'  {k} of {len(SAMPLE)} in {time.perf_counter() - t_start:.0f}s')
S4 = pd.DataFrame(rows)
S4.to_csv(C.OUT / 'wo05_fill.csv', index=False)
print(f'rows saved: {len(S4)} | this run took {time.perf_counter() - t_start:.0f}s')

In [ ]:
# Cell 5 — Item 1 over the sample: fill rates, how often the point's ecoregion differs, which DEM answered, the water classes
L = [f'sample points: {len(S4)}']
for u in UNITS:
    n_ok = int(S4[u].sum()); n_na = int(S4[u + '_na'].sum())
    miss = S4[(S4[u] == 0) & (S4[u + '_na'] == 0)]
    why = miss[u + '_why'].value_counts().to_dict()
    L.append(f'{u}: filled {n_ok} | not applicable {n_na} | could not fill {len(miss)}' + (f' {why}' if why else ''))
L.append('')
L.append(f"point's ecoregion differs from the basin's majority (S1.2c fires): {int(S4['eco_differs'].sum())} of {len(S4)}")
L.append('DEM that answered: ' + str(S4['dem'].value_counts(dropna=False).to_dict()))
L.append('water class: ' + str(S4['cls'].value_counts().to_dict()))
print('\n'.join(L))

In [ ]:
# Cell 6 — Item 2 evidence: what the five classes contain in the sample (ratios are upstream over local)
L = []
for cls, g in S4.groupby('cls'):
    L.append(f'{cls}: {len(g)} basins | upstream area over local area: median {g["area_ratio"].median():.1f}, max {g["area_ratio"].max():.0f} | distance to sink km: median {g["dist_sink"].median():.0f}, min {g["dist_sink"].min():.0f}')
    for col, nm in (('pr', 'precipitation'), ('ar', 'aridity'), ('tr', 'temperature')):
        v = g[col].dropna()
        if len(v):
            L.append(f'    {nm} ratio: below 0.9: {int((v < 0.9).sum())} | 0.9 to 1.1: {int(((v >= 0.9) & (v <= 1.1)).sum())} | 1.1 to 1.5: {int(((v > 1.1) & (v <= 1.5)).sum())} | above 1.5: {int((v > 1.5).sum())}')
print('\n'.join(L))

In [ ]:
# Cell 7 — Item 3a: readable names for the measures that appear in the S8.2 lists over the sample
labels = W.catalog_labels()
seen = pd.Series([v for s in S4['s82_vars'] for v in s.split(';') if v]).value_counts()
tab = pd.DataFrame({'basins': seen, 'readable': [(labels[v][0] + (' (upstream)' if labels[v][1] else '')) if v in labels else 'MISSING' for v in seen.index]})
print(f'{len(seen)} measures appear; {int((tab.readable == "MISSING").sum())} have no catalog name\n' + tab.to_string())

In [ ]:
# Cell 8 — Item 3b: ecoregion-to-Wikipedia links, article title against ecoregion name (writes docs/design/dashboard/wo05_wikilinks.csv)
import difflib, re
def norm(x): return re.sub(r'[^a-z0-9 ]', '', x.lower().replace('-', ' ')).strip()
q = conn.execute('SELECT e.eco_id, e.eco_name, w.wiki_title, w.wiki_url FROM gaz."Ecoregions2017" e LEFT JOIN public.eco_wikitext w USING (eco_id) ORDER BY e.eco_id').fetchall()
W8 = pd.DataFrame(q, columns=['eco_id', 'eco_name', 'wiki_title', 'wiki_url'])
W8['similarity'] = [round(difflib.SequenceMatcher(None, norm(a), norm(b)).ratio(), 2) if b else None for a, b in zip(W8.eco_name, W8.wiki_title)]
W8['title_words_in_name'] = [bool(b) and set(norm(b).split()) <= set(norm(a).split()) for a, b in zip(W8.eco_name, W8.wiki_title)]
W8['suspect'] = W8.wiki_title.notna() & ((W8.similarity < 0.6) | (W8.title_words_in_name & (W8.similarity < 0.9)))
W8.to_csv(ROOT / 'docs' / 'design' / 'dashboard' / 'wo05_wikilinks.csv', index=False)
sus = W8[W8.suspect].sort_values('similarity')
print(f"ecoregions {len(W8)} | with a link {int(W8.wiki_title.notna().sum())} | no link {int(W8.wiki_title.isna().sum())} | suspect {len(sus)} (similarity under 0.6, or the title is a part of the name); the 40 weakest matches follow, all rows are in the CSV\n" + sus[['eco_id', 'eco_name', 'wiki_title', 'similarity']].head(40).to_string())

In [ ]:
# Cell 9 — Item 3c: Alexandria's karst (kar_pc_sse is the basin share in percent; kar_pc_use is the share of the whole upstream area)
hid = TEST['Alexandria'][0]['hid']
row = conn.execute('SELECT hybas_id, kar_pc_sse, kar_pc_use, sub_area, up_area, ele_mt_sav, pre_mm_syr, coast FROM public.basin06 WHERE hybas_id = %s', (hid,)).fetchone()
nb = conn.execute("""SELECT n.hybas_id, n.kar_pc_sse FROM public.basin06 a JOIN public.basin06 n ON n.hybas_id <> a.hybas_id AND ST_Touches(a.geom, n.geom) WHERE a.hybas_id = %s ORDER BY n.hybas_id""", (hid,)).fetchall()
dist = conn.execute('SELECT count(*), count(*) FILTER (WHERE kar_pc_sse >= 99.5), count(*) FILTER (WHERE kar_pc_sse >= 50), count(*) FILTER (WHERE kar_pc_sse = 0), count(*) FILTER (WHERE kar_pc_sse < 0) FROM public.basin06').fetchone()
print('Alexandria basin06: hybas_id, kar_pc_sse, kar_pc_use, sub_area, up_area, ele_mt_sav, pre_mm_syr, coast =', row, '\nsignature karst:', W.sigval(TEST['Alexandria'][0]['pub'], 'karst'), '\nneighbours (hybas_id, kar_pc_sse):', nb,
      '\nall L6 basins: total, karst >= 99.5, >= 50, = 0, negative (nodata) =', dist)

In [ ]:
# Cell 10 — Item 3d: Tbilisi's river chain to its sink, and how the source codes the Caspian
hid = TEST['Tbilisi'][0]['hid']
chain, cur = [], hid
while cur and len(chain) < 60:
    r = conn.execute('SELECT hybas_id, next_down, next_sink, main_bas, endo, coast, dist_sink::float8, sub_area::float8 FROM public.basin06 WHERE hybas_id = %s', (cur,)).fetchone()
    chain.append(r); cur = r[1]
C10 = pd.DataFrame(chain, columns=['hybas_id', 'next_down', 'next_sink', 'main_bas', 'endo', 'coast', 'dist_sink', 'sub_area'])
sink = chain[0][2]
same = conn.execute('SELECT count(*), min(dist_sink::float8), max(dist_sink::float8) FROM public.basin06 WHERE next_sink = %s', (sink,)).fetchone()
endo = conn.execute('SELECT endo, count(*) FROM public.basin06 WHERE next_sink = %s GROUP BY endo ORDER BY endo', (sink,)).fetchall()
print(C10.to_string() + f'\n\nbasins whose next_sink is Tbilisi\'s ({sink}): {same[0]} | dist_sink from {same[1]} to {same[2]} km | endo counts: {endo}')